In [1]:
from ngo.numericalanalysis.basisfunctions import *
from ngo.numericalanalysis.quadrature import *

In [8]:
N_samples = 100
n_t = 2
n_x = 10
n_y = 10
N = n_t * n_x * n_y
Q_t = 3
Q_x = 99
Q_y = 99

n_elements_t = 1
n_elements_x = 3
n_elements_y = 3

basis_t = BSplineBasis1D(n=n_t, p=1, C=0)
basis_x = BSplineBasis1D(n=n_x, p=3, C=2)
basis_y = BSplineBasis1D(n=n_y, p=3, C=2)

quadrature_t = GaussLegendreQuadrature(Q=[Q_t], n_elements=[n_elements_t])
quadrature_x = GaussLegendreQuadrature(Q=[Q_x], n_elements=[n_elements_x])
quadrature_y = GaussLegendreQuadrature(Q=[Q_y], n_elements=[n_elements_y])
t = quadrature_t.xi.reshape(quadrature_t.xi.shape[0],)
x = quadrature_x.xi.reshape(quadrature_x.xi.shape[0],) 
y = quadrature_y.xi.reshape(quadrature_y.xi.shape[0],)

X = t,x,y

quadrature = GaussLegendreQuadrature(Q=(Q_t,Q_x,Q_y), n_elements=(n_elements_t,n_elements_x,n_elements_y))
X_q = quadrature.xi

u_m = torch.rand(size=(N_samples,N), device='cuda', dtype=torch.float64)
U = u_m.reshape((N_samples, n_t, n_x, n_y))


In [9]:
# class TensorProductBasis:
#     """
#     Tensor-product (Kronecker) basis built from multiple 1D basis objects.

#     This class combines several 1D basis objects into a d-dimensional basis by
#     taking tensor products of the 1D basis functions. It evaluates a field
#     given modal coefficients `u_m` at tensor-product evaluation points and
#     computes partial derivatives by replacing a 1D basis with its derivative
#     in the corresponding dimension.

#     Requirements for each 1D basis object:
#       - .n attribute: number of basis functions in that 1D basis
#       - .forward(xi): returns shape (n_basis_i, n_points_i)
#       - .grad(xi): returns shape (n_basis_i, n_points_i)

#     Attributes:
#         bases (tuple): Sequence of 1D basis objects.
#         d (int): Number of dimensions (len(bases)).
#         h (tuple): Number of basis functions per dimension (h1, ..., hd).
#     """
#     def __init__(self, bases):
#         """
#         Initialize the tensorized basis.

#         Args:
#             bases (list): A list of 1D basis objects.
#         """
#         self.bases = bases
#         self.d = len(bases)
#         self.n = ()
#         for i in range(len(bases)):
#             self.n += (bases[i].n,)

#     def forward(self, u_m, x):
#         """
#         Evaluate the tensor-product expansion at provided points.

#         Args:
#             u_m (np.ndarray): Modal coefficients with shape (N_samples, prod(h)) or
#                               (N_samples, h1, h2, ..., hd). N_samples is batch size.
#             x (tuple/list): Sequence of point arrays for each dimension.
#                             x[i] has shape (n_points_i,).

#         Returns:
#             np.ndarray: Evaluated values at tensor-product grid points,
#                         shape (N, n_points_1, ..., n_points_d).
#         """
#         # reshape coefficients into (N_samples, h1, h2, ..., hd)
#         u_q = u_m.reshape(u_m.shape[0],*self.n)
#         # sequentially contract coefficient tensor with each 1D basis evaluation
#         for i in range(self.d):
#            # compute basis evaluation matrix of dimension i
#             Phi_i = self.bases[i].forward(x[i])
#             # Add batch dimension if missing
#             if Phi_i.ndim == 2:
#                 Phi_i = Phi_i[None,:,:]
#             # contract along the i-th dimension
#             u_q = u_q.moveaxis(1, -1)  # move basis axis to the end for einsum
#             u_q = torch.einsum('N...n,Nnq->N...q', u_q, Phi_i)
#         # final shape is (N_samples, n_points_1, ..., n_points_d)
#         return u_q

#     def grad(self, u_m, x):
#         """
#         Compute partial derivatives of the tensor-product expansion.
#         For each spatial dimension i the method replaces the 1D basis eval in
#         that dimension with its derivative and contracts as in `forward`.

#         Args:
#             u_m (np.ndarray): Modal coefficients, shape (N, prod(h)) or (N, h1, ..., hd).
#             x (tuple/list): Sequence of point arrays for each dimension.
#                             x[i] has shape (n_points_i,).
#         Returns:
#             np.ndarray: Gradients at tensor-product points with shape
#                         (N, n_points_1, ..., n_points_d, d). The last axis indexes
#                         partial derivatives w.r.t. each coordinate.
#         """
#         #Compute function evaluation and gradient matrices
#         Phi_i = tuple(basis.forward(xi) for basis, xi in zip(self.bases, x))
#         dPhi_i = tuple(basis.grad(xi) for basis, xi in zip(self.bases, x))
#         # Add batch dimension if missing
#         Phi_i = tuple(Phi[None,:,:] if Phi.ndim==2 else Phi for Phi in Phi_i)
#         dPhi_i = tuple(dPhi[None,:,:] if dPhi.ndim==2 else dPhi for dPhi in dPhi_i)
#         # Compute derivative for each dimension by contracting with dPhi in that dim.
#         gradu_q = ()
#         for i in range(self.d):
#             # start from coefficients reshaped to (N, h1, ..., hd)
#             u_q = u_m.reshape(u_m.shape[0],*self.n)
#             for j in range(self.d):
#                 u_q = u_q.moveaxis(1, -1)  # move basis axis to the end for einsum
#                 if j==i:
#                     # use derivative matrix in the i-th dimension
#                     u_q = torch.einsum('N...n,Nnq->N...q', u_q, dPhi_i[j])
#                 else:
#                     # use value matrix in other dimensions
#                     u_q = torch.einsum('N...n,Nnq->N...q', u_q, Phi_i[j])
#             # append gradient in direction i to gradu_q tuple
#             gradu_q += (u_q,)
#         #Reshape such that the last axis indexes the coordinate to which derivative has been taken
#         gradu_q = torch.stack(gradu_q, axis=-1)
#         return gradu_q

In [14]:
class TensorProductBasis:
    """
    Tensor-product (Kronecker) basis built from multiple 1D basis objects.

    This class combines several 1D basis objects into a d-dimensional basis by
    taking tensor products of the 1D basis functions. It evaluates a field
    given modal coefficients `u_m` at tensor-product evaluation points and
    computes partial derivatives by replacing a 1D basis with its derivative
    in the corresponding dimension.

    Requirements for each 1D basis object:
      - .n attribute: number of basis functions in that 1D basis
      - .forward(xi): returns shape (n_basis_i, n_points_i)
      - .grad(xi): returns shape (n_basis_i, n_points_i)

    Attributes:
        bases (tuple): Sequence of 1D basis objects.
        d (int): Number of dimensions (len(bases)).
        n (tuple): Number of basis functions per dimension (n1, ..., nd).
    """
    def __init__(self, bases):
        """
        Initialize the tensorized basis.

        Args:
            bases (list): A list of 1D basis objects.
        """
        self.bases = bases
        self.d = len(bases)
        self.n = ()
        for i in range(len(bases)):
            self.n += (bases[i].n,)

    def Phi(self, X):
        """
        Evaluate the 1D basis functions at provided points.
        
        Args:
            X (tuple/list): Sequence of point arrays for each dimension.
                            X[i] has shape (n_points_i,).
        Returns:
            tuple: Sequence of basis evaluation matrices for each dimension.
                   Each element has shape (n_basis_i, n_points_i).
        """
        # Construct basis evaluations
        Phi = ()
        # For each dimension, compute basis evaluation matrix
        for i in range(self.d):
            Phi += (torch.tensor(self.bases[i].forward(X[i]), device='cuda', dtype=torch.float64),)
        # Add batch dimension if missing
        Phi = tuple(Phi_i[None,:,:] if Phi_i.ndim==2 else Phi_i for Phi_i in Phi)
        return Phi
    
    def dPhi(self, X):
        """
        Evaluate the derivatives of the 1D basis functions at provided points.

        Args:
            X (tuple/list): Sequence of point arrays for each dimension.
                            X[i] has shape (n_points_i,).
        Returns:
            tuple: Sequence of basis derivative matrices for each dimension.
                   Each element has shape (n_basis_i, n_points_i).
        """
        # Construct basis derivatives
        dPhi = ()
        # For each dimension, compute basis derivative matrix
        for i in range(self.d):
            dPhi += (torch.tensor(self.bases[i].grad(X[i]), device='cuda', dtype=torch.float64),)
        # Add batch dimension if missing
        dPhi = tuple(dPhi_i[None,:,:] if dPhi_i.ndim==2 else dPhi_i for dPhi_i in dPhi)
        return dPhi

    # def gradPhi(self, X):
    #     """
    #     Compute the gradient of the tensor-product basis functions at provided points.

    #     Args:
    #         X (tuple/list): Sequence of point arrays for each dimension.
    #                         X[i] has shape (n_points_i,).
    #     Returns:
    #         tuple: Sequence of gradient tuples for each dimension.
    #                Each gradient tuple contains basis evaluation matrices,
    #                with the derivative in the corresponding dimension.
    #     """
    #     # Compute basis evaluations and derivatives
    #     Phi = self.Phi(X)
    #     Phi = tuple(Phi_i[:,None] for Phi_i in Phi)  # add extra dim for broadcasting
    #     dPhi = self.dPhi(X)
    #     # Construct gradient tuples
    #     gradPhi = ()
    #     # For each dimension, create a tuple with dPhi in that dimension and Phi elsewhere
    #     for i in range(self.d):
    #         # Construct gradient in dimension i
    #         gradPhi_i = torch.ones_like(Phi[0])
    #         print(gradPhi_i.shape)
    #         # Multiply basis evaluations, using derivative in dimension i
    #         for j in range(self.d):
    #             # Use derivative in dimension i, value elsewhere
    #             if i==j:
    #                 print(dPhi[j].shape)
    #                 gradPhi_i *= dPhi[j]
    #             else:
    #                 print(Phi[j].shape)
    #                 gradPhi_i *= Phi[j]
    #         # Append gradient tuple for dimension i
    #         gradPhi += (gradPhi_i,)
    #     return gradPhi

    def forward(self, u_m, X):
        """
        Evaluate the tensor-product expansion at provided points.

        Args:
            u_m (np.ndarray): Modal coefficients with shape (N_samples, prod(n)) or
                              (N_samples, n1, n2, ..., nd). N_samples is batch size.
            X (tuple/list): Sequence of point arrays for each dimension.
                            X[i] has shape (n_points_i,).

        Returns:
            np.ndarray: Evaluated values at tensor-product grid points,
                        shape (N, n_points_1, ..., n_points_d).
        """
        # reshape coefficients into (N_samples, n1, n2, ..., nd)
        u_q = u_m.reshape(u_m.shape[0],*self.n)
        # Compute basis evaluation matrices
        Phi = self.Phi(X)
        # sequentially contract coefficient tensor with each 1D basis evaluation
        for i in range(self.d):
            # contract along the i-th dimension
            u_q = u_q.moveaxis(1, -1)  # move basis axis to the end for einsum
            u_q = torch.einsum('N...n,Nnq->N...q', u_q, Phi[i])
        # final shape is (N_samples, n_points_1, ..., n_points_d)
        return u_q

    def grad(self, u_m, X):
        """
        Compute partial derivatives of the tensor-product expansion.
        For each spatial dimension i the method replaces the 1D basis eval in
        that dimension with its derivative and contracts as in `forward`.

        Args:
            u_m (np.ndarray): Modal coefficients, shape (N, prod(n)) or (N, n1, ..., nd).
            X (tuple/list): Sequence of point arrays for each dimension.
                            x[i] has shape (n_points_i,).
        Returns:
            np.ndarray: Gradients at tensor-product points with shape
                        (N, n_points_1, ..., n_points_d, d). The last axis indexes
                        partial derivatives w.r.t. each coordinate.
        """
        #Compute function evaluation and gradient matrices
        Phi = self.Phi(X)
        dPhi = self.dPhi(X)
        # Compute derivative for each dimension by contracting with dPhi in that dim.
        gradu_q = ()
        for i in range(self.d):
            # start from coefficients reshaped to (N, h1, ..., hd)
            u_q = u_m.reshape(u_m.shape[0],*self.n)
            for j in range(self.d):
                u_q = u_q.moveaxis(1, -1)  # move basis axis to the end for einsum
                if j==i:
                    # use derivative matrix in the i-th dimension
                    u_q = torch.einsum('N...n,Nnq->N...q', u_q, dPhi[j])
                else:
                    # use value matrix in other dimensions
                    u_q = torch.einsum('N...n,Nnq->N...q', u_q, Phi[j])
            # append gradient in direction i to gradu_q tuple
            gradu_q += (u_q,)
        #Reshape such that the last axis indexes the coordinate to which derivative has been taken
        gradu_q = torch.stack(gradu_q, axis=-1)
        return gradu_q

In [15]:
class TensorizedBasis:
    """
    A class to represent tensorized basis functions.
    This class combines multiple 1D basis functions into a higher-dimensional tensorized/Kronecker product basis.

    Attributes:
        bases (list): A list of 1D basis objects.
        d (int): Dimensionality of the tensorized basis (number of 1D bases).
        n_basisfunctions (int): Total number of basis functions in the tensorized basis.
    """
    def __init__(self, bases):
        """
        Initialize the tensorized basis.

        Args:
            bases (list): A list of 1D basis objects.
        """
        self.bases = bases
        self.d = len(bases)
        self.n_basisfunctions = 1
        for i in range(len(bases)):
            self.n_basisfunctions *= bases[i].n
    
    def forward(self, x):
        """
        Compute the tensorized basis values at given points.

        Args:
            x (np.ndarray): Input points. Shape: (n_points, d).

        Returns:
            np.ndarray: Tensorized basis values at the input points. Shape: (n_points, n_basisfunctions).
        """
        tensorproduct = self.bases[0].forward(x[:,0]).T
        newsize = self.bases[0].n
        for i in range(1,self.d):
            newsize = newsize*self.bases[i].n
            tensorproduct = opt_einsum.contract('Nm,Nn->Nmn', tensorproduct, self.bases[i].forward(x[:,i]).T).reshape(x.shape[0], newsize)
        return tensorproduct
    
    def grad(self, x):
        """
        Compute the gradients of the tensorized basis functions.

        Args:
            x (np.ndarray): Input points. Shape: (n_points, d).
            
        Returns:
            np.ndarray: Gradients of the tensorized basis functions at the input points.
                        Shape: (n_points, n_basisfunctions, d).
        """
        tensorproduct = np.zeros((x.shape[0], self.n_basisfunctions, self.d))
        for i in range(self.d):
            if i==0:
                tensorproduct_i = self.bases[0].grad(x[:,0]).T
            else:
                tensorproduct_i = self.bases[0].forward(x[:,0]).T
            newsize = self.bases[0].n
            for j in range(1,self.d):
                newsize = newsize*self.bases[j].n
                if j==i:
                    tensorproduct_i = opt_einsum.contract('Nm,Nn->Nmn', tensorproduct_i, self.bases[j].grad(x[:,j]).T).reshape(x.shape[0], newsize)
                else: 
                    tensorproduct_i = opt_einsum.contract('Nm,Nn->Nmn', tensorproduct_i, self.bases[j].forward(x[:,j]).T).reshape(x.shape[0], newsize)
            tensorproduct[:,:,i] = tensorproduct_i
        return tensorproduct

In [16]:
basis1 = TensorProductBasis(bases=(basis_t,basis_x,basis_y))

In [17]:
result1 = basis1.forward(u_m, X)

In [18]:
result1.shape

torch.Size([100, 3, 99, 99])

In [19]:
basis2 = TensorizedBasis(bases=[basis_t,basis_x,basis_y])

In [20]:
Phi2 = torch.tensor(basis2.forward(X_q), device='cuda', dtype=torch.float64)

In [21]:
result2 = torch.einsum('Nn,qn->Nq', u_m, Phi2).reshape(N_samples,Q_t,Q_x,Q_y)

In [22]:
torch.amax(torch.abs(result2 - result1))

tensor(7.7716e-16, device='cuda:0', dtype=torch.float64)

In [23]:
grad1 = basis1.grad(u_m, X)

In [24]:
grad1.shape

torch.Size([100, 3, 99, 99, 3])

In [25]:
dPhi2 = torch.tensor(basis2.grad(X_q), device='cuda', dtype=torch.float64)

In [26]:
dPhi2.shape

torch.Size([29403, 200, 3])

In [27]:
grad2 = torch.einsum('Nn,qnx->Nqx', u_m, dPhi2).reshape(N_samples,Q_t,Q_x,Q_y,3)

In [28]:
torch.amax(torch.abs(grad2 - grad1))

tensor(1.4211e-14, device='cuda:0', dtype=torch.float64)